# CICIoV2024 – Veri Keşfi ve Sızıntı Analizi (Aşama 3)

Bu not defteri, deney tasarımından önce şu soruları yanıtlar:
1. Sınıf dağılımı nasıl?
2. Zaman damgası var mı?
3. Tam satır ve öznitelik düzeyinde **tekrar oranı** ne kadar?
4. Aynı öznitelik vektörü **farklı etiketlerle** görünüyor mu (sınıflar arası çakışma)?
5. Rastgele %80/20 bölmede test örneklerinin yüzde kaçı eğitim kümesinde **birebir** var? (sızıntı ölçüsü)
6. Saldırılar belirli CAN ID'lerine mi yoğunlaşıyor? (kısayol öğrenme riski)

**Veri:** https://www.unb.ca/cic/datasets/iov-dataset-2024.html adresinden indir, **decimal** formatındaki CSV dosyalarını Google Drive'da bir klasöre koy ve aşağıdaki `DATA_DIR` yolunu güncelle.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/CICIoV2024/decimal'   # <-- kendi klasörünle değiştir

In [ ]:
import glob, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.width', 200)

files = sorted(glob.glob(os.path.join(DATA_DIR, '*.csv')))
print(len(files), 'dosya bulundu')
frames = []
for f in files:
    d = pd.read_csv(f)
    d['source_file'] = os.path.basename(f)
    print(f'{os.path.basename(f):45s} {d.shape}')
    frames.append(d)
df = pd.concat(frames, ignore_index=True)
df.columns = [c.strip() for c in df.columns]
print('\nToplam:', df.shape)
print(df.columns.tolist())
df.head()

## 1. Öznitelik ve etiket sütunlarını belirle

In [ ]:
known_labels = [c for c in df.columns if c.lower() in ('label', 'category', 'specific_class', 'class', 'attack')]
label_cols = known_labels or [c for c in df.columns if df[c].dtype == object and c != 'source_file']
feature_cols = [c for c in df.columns if c not in label_cols + ['source_file']]
time_cols = [c for c in df.columns if 'time' in c.lower() or 'stamp' in c.lower()]

print('Etiket sütunları   :', label_cols)
print('Öznitelik sütunları:', feature_cols)
print('Zaman sütunları    :', time_cols or 'YOK -> zamana dayalı bölme yapılamaz, bildiride kısıt olarak yazılacak')

TARGET = next((c for c in ['specific_class', 'category', 'label'] if c in label_cols), label_cols[-1])
print('Hedef etiket:', TARGET)

## 2. Sınıf dağılımı

In [ ]:
for c in label_cols:
    vc = df[c].value_counts()
    print(f'--- {c} ---')
    print(pd.DataFrame({'adet': vc, 'oran_%': (vc / len(df) * 100).round(3)}), '\n')

df[TARGET].value_counts().plot.bar(logy=True, title='Sınıf dağılımı (log ölçek)')
plt.ylabel('Satır sayısı'); plt.tight_layout(); plt.show()

## 3. Tekrar analizi

In [ ]:
n = len(df)
dup_full = df.duplicated(subset=feature_cols + [TARGET]).sum()
dup_feat = df.duplicated(subset=feature_cols).sum()
print(f'Toplam satır                       : {n:,}')
print(f'Tam tekrar (öznitelik+etiket)       : {dup_full:,}  (%{dup_full/n*100:.2f})')
print(f'Öznitelik vektörü tekrarı           : {dup_feat:,}  (%{dup_feat/n*100:.2f})')
print(f'Benzersiz (öznitelik+etiket) satır  : {n-dup_full:,}')

per_class = df.groupby(TARGET).apply(
    lambda g: pd.Series({'satır': len(g), 'benzersiz': len(g.drop_duplicates(subset=feature_cols))})).astype(int)
per_class['tekrar_%'] = ((1 - per_class['benzersiz'] / per_class['satır']) * 100).round(2)
per_class

## 4. Sınıflar arası çakışma
Aynı öznitelik vektörü birden fazla etiketle görünüyorsa, bu satırları hiçbir model ayırt edemez.

In [ ]:
g = df.groupby(feature_cols)[TARGET].nunique()
conflict_keys = g[g > 1].index
conflict_mask = df.set_index(feature_cols).index.isin(conflict_keys)
print(f'Çakışan benzersiz vektör sayısı : {len(conflict_keys):,}')
print(f'Çakışmadan etkilenen satır      : {conflict_mask.sum():,}  (%{conflict_mask.mean()*100:.2f})')
if conflict_mask.any():
    print('\nÇakışan satırların sınıf dağılımı:')
    print(df.loc[conflict_mask, TARGET].value_counts())

## 5. Sızıntı ölçüsü: rastgele bölmede test–eğitim örtüşmesi
Literatürdeki tipik rastgele %80/20 bölmeyi taklit eder; test satırlarının yüzde kaçının eğitim kümesinde **birebir** bulunduğunu 5 farklı tohumla ölçer.

In [ ]:
from sklearn.model_selection import train_test_split

rows = []
for seed in range(5):
    tr, te = train_test_split(df, test_size=0.2, stratify=df[TARGET], random_state=seed)
    train_keys = set(map(tuple, tr[feature_cols].to_numpy()))
    seen = np.fromiter((tuple(r) in train_keys for r in te[feature_cols].to_numpy()), bool, len(te))
    te = te.assign(seen=seen)
    r = te.groupby(TARGET)['seen'].mean().mul(100).round(2).to_dict()
    r['GENEL'] = round(seen.mean() * 100, 2)
    r['seed'] = seed
    rows.append(r)
leak = pd.DataFrame(rows).set_index('seed')
print('Test satırlarının eğitimde birebir bulunma oranı (%):')
leak.agg(['mean', 'std']).T.round(2)

## 6. CAN ID analizi (kısayol öğrenme testi)
Saldırılar belirli ID'lere yoğunlaşıyorsa, model yalnızca ID'ye bakarak sınıflandırıyor olabilir.

In [ ]:
id_col = next((c for c in feature_cols if c.upper() == 'ID'), None)
if id_col:
    print('Toplam farklı CAN ID:', df[id_col].nunique())
    tab = df.groupby(TARGET)[id_col].agg(
        farkli_ID='nunique',
        en_sik_ID=lambda s: s.value_counts().index[0],
        en_sik_ID_orani=lambda s: round(s.value_counts(normalize=True).iloc[0] * 100, 2))
    display(tab)
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.metrics import f1_score
    dd = df.drop_duplicates(subset=feature_cols + [TARGET])
    tr, te = train_test_split(dd, test_size=0.2, stratify=dd[TARGET], random_state=0)
    clf = DecisionTreeClassifier(random_state=0).fit(tr[[id_col]], tr[TARGET])
    print('Yalnızca CAN ID ile makro F1 (tekrarsız veri):',
          round(f1_score(te[TARGET], clf.predict(te[[id_col]]), average='macro'), 4))
else:
    print('ID sütunu bulunamadı; sütun adlarını kontrol et.')

## 7. Özet tablo
Çıktıyı bana gönder; deney tasarımını bu sonuçlara göre kesinleştireceğiz.

In [ ]:
summary = pd.DataFrame({
    'metrik': ['toplam_satir', 'tam_tekrar_%', 'oznitelik_tekrar_%', 'cakisan_satir_%',
               'rastgele_bolme_sizinti_%', 'zaman_damgasi_var'],
    'deger': [n, round(dup_full/n*100, 2), round(dup_feat/n*100, 2), round(conflict_mask.mean()*100, 2),
              round(leak['GENEL'].mean(), 2), bool(time_cols)]})
summary.to_csv('/content/ciciov2024_kesif_ozeti.csv', index=False)
per_class.to_csv('/content/ciciov2024_sinif_tekrar.csv')
summary